<a href="https://colab.research.google.com/github/munizgui/CP4_SERS/blob/main/CP4_SERS_EX5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Dataset 5 — Wind & Solar Energy Production (Kaggle)

Fonte: https://www.kaggle.com/datasets/ahmeduzaki/wind-and-solar-energy-production-dataset

Situação

Um operador de um portfólio de fontes renováveis deseja comparar a ocorrência de períodos de alta produção solar e alta produção eólica. Como as duas fontes possuem escalas próprias, cada uma deverá ser comparada com o seu próprio valor máximo.

In [ ]:
import pandas as pd

In [ ]:
dados = pd.read_csv("Energy Production Dataset.csv")

In [ ]:
dados.head(10)

,Date,Start_Hour,End_Hour,Source,Day_of_Year,Day_Name,Month_Name,Season,Production
0,12/18/2022,21,22,Wind,352,Sunday,December,Winter,11400
1,1/29/2024,14,15,Wind,29,Monday,January,Winter,7917
2,8/1/2024,15,16,Solar,214,Thursday,August,Summer,8835
3,11/10/2020,1,2,Wind,315,Tuesday,November,Fall,989
4,12/19/2022,1,2,Wind,353,Monday,December,Winter,12526
5,10/18/2024,13,14,Solar,292,Friday,October,Fall,4855
6,4/10/2020,17,18,Solar,101,Friday,April,Spring,3692
7,1/25/2023,0,1,Wind,25,Wednesday,January,Winter,2156
8,12/12/2021,12,13,Wind,346,Sunday,December,Winter,5813
9,10/11/2020,13,14,Wind,285,Sunday,October,Fall,6106


In [ ]:
dados.shape

(10373, 9)

In [ ]:
dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10373 entries, 0 to 10372
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Date         10373 non-null  object
 1   Start_Hour   10373 non-null  int64 
 2   End_Hour     10373 non-null  int64 
 3   Source       10373 non-null  object
 4   Day_of_Year  10373 non-null  int64 
 5   Day_Name     10373 non-null  object
 6   Month_Name   10373 non-null  object
 7   Season       10373 non-null  object
 8   Production   10373 non-null  int64 
dtypes: int64(4), object(5)
memory usage: 729.5+ KB


In [ ]:
dados.describe()

,Start_Hour,End_Hour,Day_of_Year,Production
count,10373.000000,10373.000000,10373.000000,10373.000000
mean,11.530898,11.489733,181.003181,6193.352164
std,6.947598,6.949144,103.655309,3981.573520
min,0.000000,0.000000,1.000000,91.000000
25%,6.000000,5.000000,92.000000,3117.000000
50%,12.000000,11.000000,180.000000,5334.000000
75%,18.000000,18.000000,271.000000,8443.000000
max,23.000000,23.000000,366.000000,22929.000000


In [ ]:
# 1. Renomeie as variáveis principais para Geracao_Solar e Geracao_Eolica
if 'Source' in dados.columns and 'Production' in dados.columns:
    tempo_cols = [c for c in ['Date', 'Start_Hour', 'End_Hour', 'Day_of_Year', 'Day_Name', 'Month_Name', 'Season'] if c in dados.columns]
    dados = dados.pivot_table(
        index=tempo_cols,
        columns='Source',
        values='Production',
        aggfunc='mean'
    ).reset_index()

cols_rename = {}
for col in dados.columns:
    c = str(col).lower()
    if 'solar' in c or 'sun' in c:
        cols_rename[col] = 'Geracao_Solar'
    elif 'wind' in c or 'eolica' in c or 'eólica' in c:
        cols_rename[col] = 'Geracao_Eolica'

dados.rename(columns=cols_rename, inplace=True)
dados['Geracao_Solar'] = pd.to_numeric(dados['Geracao_Solar'], errors='coerce')
dados['Geracao_Eolica'] = pd.to_numeric(dados['Geracao_Eolica'], errors='coerce')

In [ ]:
dados.columns

Index(['Date', 'Start_Hour', 'End_Hour', 'Day_of_Year', 'Day_Name',
       'Month_Name', 'Season', 'Geracao_Solar', 'Geracao_Eolica'],
      dtype='object', name='Source')

In [ ]:
# 2. Determine o valor máximo registrado para cada fonte de forma independente
max_solar = dados['Geracao_Solar'].max()
max_eolica = dados['Geracao_Eolica'].max()

print(f"O maior valor registrado para Geração Solar foi: {max_solar}")
print(f"O maior valor registrado para Geração Eólica foi: {max_eolica}")

O maior valor registrado para Geração Solar foi: 16316.0
O maior valor registrado para Geração Eólica foi: 22929.0


In [ ]:
# 3. Calcule separadamente 70% do máximo solar e 70% do máximo eólico usando as escalas próprias
limiar_solar = max_solar * 0.70
limiar_eolica = max_eolica * 0.70

print(f"Limiar (70%) Solar: {limiar_solar:.2f}")
print(f"Limiar (70%) Eólico: {limiar_eolica:.2f}")

Limiar (70%) Solar: 11421.20
Limiar (70%) Eólico: 16050.30


In [ ]:
# 4. Crie um DataFrame para registros de alta geração solar e outro para registros de alta geração eólica
df_alta_solar = dados[dados['Geracao_Solar'] >= limiar_solar]
df_alta_eolica = dados[dados['Geracao_Eolica'] >= limiar_eolica]

print("Registros de alta geração solar:")
df_alta_solar.head()

Registros de alta geração solar:


Source,Date,Start_Hour,End_Hour,Day_of_Year,Day_Name,Month_Name,Season,Geracao_Solar,Geracao_Eolica
971,10/12/2025,13,14,285,Sunday,October,Fall,12629.0,NaN
972,10/12/2025,15,16,285,Sunday,October,Fall,11438.0,NaN
1010,10/13/2025,14,15,286,Monday,October,Fall,12354.0,NaN
4430,3/17/2025,10,11,76,Monday,March,Spring,11786.0,NaN
4497,3/19/2025,13,14,78,Wednesday,March,Spring,13786.0,NaN


In [ ]:
print("Registros de alta geração eólica:")
df_alta_eolica.head()

Registros de alta geração eólica:


Source,Date,Start_Hour,End_Hour,Day_of_Year,Day_Name,Month_Name,Season,Geracao_Solar,Geracao_Eolica
93,1/12/2023,8,9,12,Thursday,January,Winter,NaN,16345.0
94,1/12/2023,9,10,12,Thursday,January,Winter,NaN,16699.0
95,1/12/2023,12,13,12,Thursday,January,Winter,NaN,18306.0
96,1/12/2023,13,14,12,Thursday,January,Winter,NaN,18143.0
97,1/12/2023,15,16,12,Thursday,January,Winter,NaN,16900.0


In [ ]:
# 5. Conte os registros de cada DataFrame e calcule seus respectivos percentuais
total_registros = len(dados)

qtd_solar = len(df_alta_solar)
pct_solar = (qtd_solar / total_registros) * 100

qtd_eolica = len(df_alta_eolica)
pct_eolica = (qtd_eolica / total_registros) * 100

print(f"Total de registros na amostra: {total_registros}")
print(f"Registros de Alta Geração Solar: {qtd_solar} ({pct_solar:.2f}%)")
print(f"Registros de Alta Geração Eólica: {qtd_eolica} ({pct_eolica:.2f}%)")

Total de registros na amostra: 10373
Registros de Alta Geração Solar: 45 (0.43%)
Registros de Alta Geração Eólica: 245 (2.36%)


In [ ]:
# 6. Compare qual fonte aparece com maior frequência acima de 70% do seu próprio máximo
if pct_solar > pct_eolica:
    print("A fonte SOLAR aparece com maior frequência acima de 70% do seu próprio máximo.")
elif pct_eolica > pct_solar:
    print("A fonte EÓLICA aparece com maior frequência acima de 70% do seu próprio máximo.")
else:
    print("Ambas as fontes possuem a mesma frequência relativa acima de 70% do seu máximo.")

A fonte EÓLICA aparece com maior frequência acima de 70% do seu próprio máximo.


# 7. Explique por que não seria adequado utilizar diretamente o mesmo valor numérico de potência como limite para as duas fontes sem antes observar a escala de cada variável.

Utilizar um único valor numérico absoluto de potência, como fixar um limite de 500 MW para ambas as fontes, é inadequado porque ignora as diferenças estruturais de capacidade instalada e a dinâmica de operação da energia solar e eólica.

Como cada matriz possui sua própria escala de produção, uma usina eólica pode ter um pico de geração significativamente maior ou menor do que um parque solar no mesmo portfólio. Se aplicássemos um limite numérico igual para as duas, haveria um problema de incompatibilidade de escala, pois a fonte que tiver a menor capacidade total poderia nunca atingir o limite estipulado, aparentando um desempenho fraco mesmo quando estivesse operando em 100% do seu potencial. Além disso, ocorreria uma distorção de eficiência, já que a fonte com maior capacidade instalada dominaria a contagem de registros simplesmente pelo seu porte, e não por estar operando perto do seu limite máximo.

Portanto, para o operador do portfólio avaliar a ocorrência de alta produção de forma justa e comparável, é indispensável relativizar os dados. A utilização do limite individual de 70% do próprio valor máximo normaliza as variáveis, permitindo analisar com precisão a frequência com que cada tecnologia atinge o topo do seu desempenho operacional esperado.